# ArcGIS API for Python: Pure Referenced Raster Dataset Publishing

This notebook demonstrates how to create a registered S3 cloud storage **Raster Dataset** **by reference** using the **ArcGIS API for Python** (no local `arcpy` and no geoprocessing toolbox imports required).

### Step 1: Connect and Authenticate to ArcGIS Enterprise Portal

In [ ]:
from os import getenv
from dotenv import load_dotenv

from arcgis.gis import GIS
from arcgis.raster.analytics import get_datastores

load_dotenv()

portal_url = getenv("PORTAL_URL")
client_id = getenv("CLIENT_ID")

gis = GIS(url=portal_url, client_id=client_id)
print(f"Hello, {gis.users.me.username}")

### Step 2: Locate and Verify existing S3 Cloud Datastore

We can use the Raster Analytics `get_datastores` helper function to query registered data stores.

In [ ]:
cloudstore_name = getenv("CLOUDSTORE_NAME")

datastores = get_datastores(gis)
cloud_datastores = datastores.search(types="cloudStore")

matching_cloudstore = [ds for ds in cloud_datastores if cloudstore_name == ds.datapath][0]
if not matching_cloudstore:
    raise ValueError(f"No cloud datastore named '{cloudstore_name}' found in datastores.")

print(f"Found cloud datastore: {matching_cloudstore}")

### Step 3: Publish Referenced Raster Dataset
We use `create_service` with parameters which reference our data in our datastore, as well as proper `create_params` (`path`, `copyData: false` and `esriImageServiceSourceTypeDataset`). We then have to manually start the service.

In [ ]:
service_name = "single_raster"
raster_file = "test.tif"

# 1. Format the input referenced dataset as a string
input_raster = f"{matching_cloudstore.datapath}/{raster_file.lstrip('/')}"
print(f"Referencing data... '{input_raster}'")

# 2. Define the image service properties
create_params = {
    "name": service_name,
    "capabilities": "Image,Metadata,Mensuration", # Key
    "cacheControlMaxAge": "43200",
    "provider": "ArcObjectsRasterRendering",
    "properties": {
        "path": input_raster, # Key
        "isManaged": True,
        "isCached": False,
        "esriImageServiceSourceType": "esriImageServiceSourceTypeDataset", # Key
        "isTiledImagery": "false",
        "colormapToRGB": "false",
        "description": f"Standard referenced Image Service published by reference from Cloud Datastore '{matching_cloudstore.datapath}' path '{raster_file}'.",
        "defaultResamplingMethod": 1,
    },
    "copyData": False # Key
}

# 3. Publish Raster Dataset as image service item
print(f"\nPublishing Item... '{service_name}'")
published_item = gis.content.create_service(
    name=service_name,
    service_type="imageService",
    create_params=create_params,
    description=f"Standard referenced Image Service published by reference from Cloud Datastore '{matching_cloudstore.datapath}' path '{raster_file}'.",
    tags=["Image Service", "Single Raster"]
)

print("\nSuccess!")
print("Item Title:", published_item.title)
print("Item ID:", published_item.id)
print("Service URL:", getattr(published_item, "url", "N/A"))
print("Hosted Status:", "Hosted" in published_item.typeKeywords)

# 4. We have to manually start the service using this method
print("Service started:", [item for item in gis.admin.servers.get(function="RasterAnalytics")[0].services.list(folder="Imagery", refresh=True) if item.properties.get("serviceName") == published_item.title][0].start())